In [ ]:
# !kaggle competitions download -c playground-series-s5e3
# !unzip -u *.zip

In [1]:
from pathlib import Path
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import missingno

from sklearn import set_config
set_config(transform_output = "pandas")

from sklearn.model_selection import ShuffleSplit, KFold, StratifiedKFold
from sklearn.model_selection import cross_validate, GridSearchCV

from sklearn.feature_selection import SelectFromModel, RFECV

from sklearn.metrics import roc_auc_score

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import LinearSVC
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


KAGGLE_RUN = False
if KAGGLE_RUN:
    working_dir = Path('/kaggle/input/playground-series-s5e3')
else:
    working_dir = Path().cwd()

2025-03-15 17:34:28.169997: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-15 17:34:28.170877: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-15 17:34:28.174801: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-15 17:34:28.184109: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742056468.203619   65875 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742056468.20

In [2]:
train_df = pd.read_csv(working_dir/'train.csv', index_col='id')
test_df = pd.read_csv(working_dir/'test.csv', index_col='id')


In [3]:
train_df

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,rainfall
id,,,,,,,,,,,,
0,1,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1
1,2,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1
2,3,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1
3,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,1
4,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,24.8,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2185,361,1014.6,23.2,20.6,19.1,19.9,97.0,88.0,0.1,40.0,22.1,1
2186,362,1012.4,17.2,17.3,16.3,15.3,91.0,88.0,0.0,50.0,35.3,1
2187,363,1013.3,19.0,16.3,14.3,12.6,79.0,79.0,5.0,40.0,32.9,1


In [4]:
test_df

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed
id,,,,,,,,,,,
2190,1,1019.5,17.5,15.8,12.7,14.9,96.0,99.0,0.0,50.0,24.3
2191,2,1016.5,17.5,16.5,15.8,15.1,97.0,99.0,0.0,50.0,35.3
2192,3,1023.9,11.2,10.4,9.4,8.9,86.0,96.0,0.0,40.0,16.9
2193,4,1022.9,20.6,17.3,15.2,9.5,75.0,45.0,7.1,20.0,50.6
2194,5,1022.2,16.1,13.8,6.4,4.3,68.0,49.0,9.2,20.0,19.4
...,...,...,...,...,...,...,...,...,...,...,...
2915,361,1020.8,18.2,17.6,16.1,13.7,96.0,95.0,0.0,20.0,34.3
2916,362,1011.7,23.2,18.1,16.0,16.0,78.0,80.0,1.6,40.0,25.2
2917,363,1022.7,21.0,18.5,17.0,15.5,92.0,96.0,0.0,50.0,21.9


In [5]:
NUMERIC_COLUMNS=['day', 'pressure', 'maxtemp', 'temparature', 'mintemp', 'dewpoint', 'humidity', 'cloud', 'sunshine', 'winddirection', 'windspeed']
CATEGORIC_COLUMNS=[]
TARGET_COLUMN=['rainfall']
ALL_COLUMNS=NUMERIC_COLUMNS+CATEGORIC_COLUMNS+TARGET_COLUMN

In [6]:
# feature engineering
#  add lag, fourier features, spreads, binning, days with maxtemp< temaparature, etc.

def group_agg_merge(df, by_column, agg_column):
    grouped_df = df.groupby(by=by_column).agg(
        **{
        f'{by_column}_average_{agg_column}':(agg_column, 'mean'),
        f'{by_column}_std_{agg_column}':(agg_column, 'std'),
        f'{by_column}_skew_{agg_column}':(agg_column, 'skew'),
        f'{by_column}_median_{agg_column}':(agg_column, 'median'),
        f'{by_column}_min_{agg_column}':(agg_column, 'min'),
        f'{by_column}_max_{agg_column}':(agg_column, 'max'),}
    )
    df = df.join(grouped_df, on=by_column, how='left')
    return df

def add_features(df):
    df=df.fillna(0)
    df['year'] = df.index//365

    df['temp_spread'] = df['maxtemp'] - df['mintemp']
    df['temp_diff_dew'] = df['temparature'] - df['dewpoint']

    df['pressure_diff'] = df['pressure'].diff().fillna(0)
    df['pressure_bins'] = pd.cut(df['pressure'], bins=[998, 1010, 1020, 1035], labels=list(range(3)), include_lowest=True)

    df['humidity_bins'] = pd.cut(df['humidity'], bins=[0, 50, 80,100], labels=list(range(3)), include_lowest=True)

    df['cloud_bins'] = pd.cut(df['cloud'], bins=[0, 25, 50, 75, 100], labels=list(range(4)), include_lowest=True)

    df['sunshine_bins'] = pd.cut(df['sunshine'], bins=[-0.1, 3., 6., 9., 13.], labels=list(range(4)), include_lowest=True)
    
    df['winddirection_bins'] = pd.cut(df['winddirection'], bins=[-1., 90, 180, 270, 361.], labels=list(range(4)), include_lowest=True)
    df['windspeed_bins'] = pd.cut(df['windspeed'], bins=[0, 15, 30, 45, 60], labels=list(range(4)), include_lowest=True)

    df['max_temp_under_temp'] = df['maxtemp'] < df['temparature']
    df['min_temp_over_temp'] = df['mintemp'] > df['temparature']

    df['sin_day'] = np.sin(2*np.pi* df['day']/365)
    df['cos_day'] = np.cos(2*np.pi* df['day']/365)

    # create lagged features for somecolumns
    for (column, shift) in itertools.product(['temparature', 'pressure', 'dewpoint', 'humidity', 'sunshine', 'windspeed'], range(1,3),repeat=1):
            df[f'lagged_{column}_{shift}'] = df[column].shift(shift).fillna(0)
    
    df['month'] = df['day'].apply(
        lambda x: 'Jan' if 1<=x<=31 
        else 'Feb' if 32<= x<= 59
        else 'Mar' if 60<= x<= 90
        else 'Apr' if 91<= x<= 120
        else 'May' if 121<= x<= 151
        else 'Jun' if 152<= x<= 181
        else 'Jul' if 182<= x<= 212
        else 'Aug' if 213<= x<= 243
        else 'Sep' if 244<= x<= 273
        else 'Oct' if 274<= x<= 304
        else 'Nov' if 305<= x<= 334
        else 'Dec' if 335<= x<= 366
        else 'Nan'
    )

    df['season'] = df['month'].apply(
        lambda x: 'Winter' if x in ['Dec', 'Jan', 'Feb']
        else 'Spring' if x in ['Mar', 'Apr', 'May']
        else 'Summer' if x in ['Jun', 'Jul', 'Aug']
        else 'Autumn' if x in ['Sep', 'Oct', 'Nov']
        else 'Nan'
    )

    for i, k in itertools.product(['month', 'season'], ['temparature', 'pressure', 'dewpoint', 'humidity', 'sunshine', 'windspeed']):
        df = group_agg_merge(df, i, k)

    # interaction features
    for (column1, column2) in itertools.combinations(NUMERIC_COLUMNS, 2):
        df[f'interaction_{column1}_{column2}'] = df[column1]*df[column2]

    return df



train_df = add_features(train_df)
test_df = add_features(test_df)

ADDED_NUMERIC_COLUMNS=[
    'sin_day',
    'cos_day',
    'year',
    'temp_spread',
    'temp_diff_dew',
    'pressure_diff',
    ]+[
        f'interaction_{column1}_{column2}' for (column1, column2) in itertools.combinations(NUMERIC_COLUMNS, 2)
    ]+[
        f'{i}_{j}_{k}' for i,j,k in itertools.product(['month', 'season'], ['average', 'std', 'skew', 'median', 'min', 'max'], ['temparature', 'pressure', 'dewpoint', 'humidity', 'sunshine', 'windspeed'])
    ]+[
        f'lagged_{column}_{shift}' for (column, shift) in itertools.product(['temparature', 'pressure', 'dewpoint', 'humidity', 'sunshine', 'windspeed'], range(1,3),repeat=1)
    ]
NUMERIC_COLUMNS+=ADDED_NUMERIC_COLUMNS
CATEGORIC_COLUMNS+=[
    'pressure_bins',
    'humidity_bins',
    'cloud_bins',
    'sunshine_bins',
    'winddirection_bins',
    'windspeed_bins',
    'month',
    'season',
    'max_temp_under_temp',
    'min_temp_over_temp'
    ]

In [7]:
print(NUMERIC_COLUMNS)
print(CATEGORIC_COLUMNS)

['day', 'pressure', 'maxtemp', 'temparature', 'mintemp', 'dewpoint', 'humidity', 'cloud', 'sunshine', 'winddirection', 'windspeed', 'sin_day', 'cos_day', 'year', 'temp_spread', 'temp_diff_dew', 'pressure_diff', 'interaction_day_pressure', 'interaction_day_maxtemp', 'interaction_day_temparature', 'interaction_day_mintemp', 'interaction_day_dewpoint', 'interaction_day_humidity', 'interaction_day_cloud', 'interaction_day_sunshine', 'interaction_day_winddirection', 'interaction_day_windspeed', 'interaction_pressure_maxtemp', 'interaction_pressure_temparature', 'interaction_pressure_mintemp', 'interaction_pressure_dewpoint', 'interaction_pressure_humidity', 'interaction_pressure_cloud', 'interaction_pressure_sunshine', 'interaction_pressure_winddirection', 'interaction_pressure_windspeed', 'interaction_maxtemp_temparature', 'interaction_maxtemp_mintemp', 'interaction_maxtemp_dewpoint', 'interaction_maxtemp_humidity', 'interaction_maxtemp_cloud', 'interaction_maxtemp_sunshine', 'interaction_

In [8]:
target = train_df[TARGET_COLUMN]
train = train_df.drop(columns=TARGET_COLUMN)
test = test_df

In [9]:
target

,rainfall
id,
0,1
1,1
2,1
3,1
4,0
...,...
2185,1
2186,1
2187,1


In [10]:
train

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,...,interaction_humidity_cloud,interaction_humidity_sunshine,interaction_humidity_winddirection,interaction_humidity_windspeed,interaction_cloud_sunshine,interaction_cloud_winddirection,interaction_cloud_windspeed,interaction_sunshine_winddirection,interaction_sunshine_windspeed,interaction_winddirection_windspeed
id,,,,,,,,,,,,,,,,,,,,,
0,1,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,...,7656.0,95.7,5220.0,1496.4,96.8,5280.0,1513.6,66.0,18.92,1032.0
1,2,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,...,8645.0,0.0,4750.0,2080.5,0.0,4550.0,1992.9,0.0,0.00,1095.0
2,3,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,...,3525.0,622.5,5250.0,1357.5,390.1,3290.0,850.7,581.0,150.23,1267.0
3,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,...,9025.0,0.0,5700.0,3382.0,0.0,5700.0,3382.0,0.0,0.00,2136.0
4,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,...,2340.0,187.2,2080.0,1289.6,162.0,1800.0,1116.0,144.0,89.28,992.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2185,361,1014.6,23.2,20.6,19.1,19.9,97.0,88.0,0.1,40.0,...,8536.0,9.7,3880.0,2143.7,8.8,3520.0,1944.8,4.0,2.21,884.0
2186,362,1012.4,17.2,17.3,16.3,15.3,91.0,88.0,0.0,50.0,...,8008.0,0.0,4550.0,3212.3,0.0,4400.0,3106.4,0.0,0.00,1765.0
2187,363,1013.3,19.0,16.3,14.3,12.6,79.0,79.0,5.0,40.0,...,6241.0,395.0,3160.0,2599.1,395.0,3160.0,2599.1,200.0,164.50,1316.0


In [11]:
test

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,...,interaction_humidity_cloud,interaction_humidity_sunshine,interaction_humidity_winddirection,interaction_humidity_windspeed,interaction_cloud_sunshine,interaction_cloud_winddirection,interaction_cloud_windspeed,interaction_sunshine_winddirection,interaction_sunshine_windspeed,interaction_winddirection_windspeed
id,,,,,,,,,,,,,,,,,,,,,
2190,1,1019.5,17.5,15.8,12.7,14.9,96.0,99.0,0.0,50.0,...,9504.0,0.0,4800.0,2332.8,0.0,4950.0,2405.7,0.0,0.00,1215.0
2191,2,1016.5,17.5,16.5,15.8,15.1,97.0,99.0,0.0,50.0,...,9603.0,0.0,4850.0,3424.1,0.0,4950.0,3494.7,0.0,0.00,1765.0
2192,3,1023.9,11.2,10.4,9.4,8.9,86.0,96.0,0.0,40.0,...,8256.0,0.0,3440.0,1453.4,0.0,3840.0,1622.4,0.0,0.00,676.0
2193,4,1022.9,20.6,17.3,15.2,9.5,75.0,45.0,7.1,20.0,...,3375.0,532.5,1500.0,3795.0,319.5,900.0,2277.0,142.0,359.26,1012.0
2194,5,1022.2,16.1,13.8,6.4,4.3,68.0,49.0,9.2,20.0,...,3332.0,625.6,1360.0,1319.2,450.8,980.0,950.6,184.0,178.48,388.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2915,361,1020.8,18.2,17.6,16.1,13.7,96.0,95.0,0.0,20.0,...,9120.0,0.0,1920.0,3292.8,0.0,1900.0,3258.5,0.0,0.00,686.0
2916,362,1011.7,23.2,18.1,16.0,16.0,78.0,80.0,1.6,40.0,...,6240.0,124.8,3120.0,1965.6,128.0,3200.0,2016.0,64.0,40.32,1008.0
2917,363,1022.7,21.0,18.5,17.0,15.5,92.0,96.0,0.0,50.0,...,8832.0,0.0,4600.0,2014.8,0.0,4800.0,2102.4,0.0,0.00,1095.0


In [12]:
transformer = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), NUMERIC_COLUMNS),
        ('categories', OneHotEncoder(sparse_output=False), CATEGORIC_COLUMNS),
    ], remainder='passthrough'
)

train_scaled = transformer.fit_transform(train)
test_scaled = transformer.transform(test)


In [13]:
train_scaled

,numeric__day,numeric__pressure,numeric__maxtemp,numeric__temparature,numeric__mintemp,numeric__dewpoint,numeric__humidity,numeric__cloud,numeric__sunshine,numeric__winddirection,...,categories__month_Oct,categories__month_Sep,categories__season_Autumn,categories__season_Spring,categories__season_Summer,categories__season_Winter,categories__max_temp_under_temp_False,categories__max_temp_under_temp_True,categories__min_temp_over_temp_False,categories__min_temp_over_temp_True
id,,,,,,,,,,,,,,,,,,,,,
0,-1.701361,0.671702,-0.913809,-0.642199,-0.448815,-0.199457,0.636434,0.681269,-0.729397,-0.560901,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
1,-1.691853,1.043116,-1.798289,-1.350846,-1.259418,-0.956001,1.662224,0.847728,-1.032804,-0.685925,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
2,-1.682346,1.856688,-1.232222,-1.504067,-1.496667,-2.109731,-0.902250,-1.593680,1.256536,-0.435876,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
3,-1.672838,-0.035752,-1.462187,-1.178472,-1.041939,-0.691210,1.662224,1.069675,-1.032804,-0.560901,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
4,-1.663331,1.449902,-0.896120,-1.063556,-1.378043,-2.052990,-3.851394,-1.704654,-0.039837,-0.810950,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2185,1.721357,0.176484,-0.560017,-0.642199,-0.606982,-0.104888,1.918671,0.681269,-1.005222,-0.810950,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
2186,1.730865,-0.212616,-1.621393,-1.274235,-1.160564,-0.974914,1.149329,0.681269,-1.032804,-0.685925,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
2187,1.740372,-0.053439,-1.302980,-1.465761,-1.555980,-1.485582,-0.389355,0.181890,0.346317,-0.810950,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0


In [14]:
model= Sequential([
    Dense(128, activation='relu', kernel_initializer='glorot_uniform', input_shape=(train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_initializer='glorot_uniform'),
    Dropout(0.2),
    Dense(32, activation='relu', kernel_initializer='glorot_uniform'),
    Dropout(0.1),
    Dense(16, activation='relu', kernel_initializer='glorot_uniform'),
    Dense(1, activation='sigmoid'),
]
)

optimizer = Adam(learning_rate=1e-3)

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['auc']
)

/home/mihofer/Repos/kaggle_projects/statspython/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-03-15 17:34:42.804566: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [15]:
history = model.fit(
    train_scaled,
    target,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    ],
    verbose=1
)

Epoch 1/200
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - auc: 0.6078 - loss: 0.5802 - val_auc: 0.8768 - val_loss: 0.3267
Epoch 2/200
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - auc: 0.8738 - loss: 0.3615 - val_auc: 0.8754 - val_loss: 0.3335
Epoch 3/200
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - auc: 0.8855 - loss: 0.3600 - val_auc: 0.8821 - val_loss: 0.3260
Epoch 4/200
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - auc: 0.8955 - loss: 0.3278 - val_auc: 0.8759 - val_loss: 0.3339
Epoch 5/200
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - auc: 0.8884 - loss: 0.3445 - val_auc: 0.8878 - val_loss: 0.3177
Epoch 6/200
55/55 ━━━━━━━━━━━━━━━━━━━━ -0s -8160us/step - auc: 0.8940 - loss: 0.3402 - val_auc: 0.8772 - val_loss: 0.3325
Epoch 7/200
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - auc: 0.9033 - loss: 0.3334 - val_auc: 0.8658 - val_loss: 0.3429
Epoch 8/200
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - auc: 0.9140 - loss: 0.3127 - val_auc: 0.8868 - val_loss: 0.3230
Epoch 9/200
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - auc: 

In [24]:
model.predict(test_scaled).flatten()
to_binary = np.vectorize(lambda x: 0 if x<0.5 else 1)
rainfall = to_binary(model.predict(test_scaled).flatten())
rainfall

 1/23 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


array([1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1,
       1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
       0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1,
       1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1,
       1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
       1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [25]:
sub_df = pd.DataFrame(
    index=test.index,
    data={
        'rainfall':rainfall
    },
)
sub_df    


,rainfall
id,
2190,1
2191,1
2192,1
2193,0
2194,0
...,...
2915,1
2916,1
2917,1


In [ ]:
if KAGGLE_RUN:
    sub_df.to_csv("/kaggle/working/submission.csv", index_label='id')
    !head /kaggle/working/submission.csv